# Lesson 04 — 给计时器一张状态图 — 参考实现

先完成主 Notebook 的练习再阅读。此 Notebook 只展示可通过 exercise 契约的一种实现。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


运行代码格，使用同一个可交互面板验证。然后回到主 Notebook，对照自己的状态和边界处理。


In [ ]:
%%academy_lab 04-state-machine solution exercise pass
#pragma once
#include "academy/device.hpp"
#include <cstdint>
#include <string>
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& board);
        void tick();

    private:
        Device& board_;
        std::string last_command_;
        std::string state_ = "idle";
        std::uint32_t end_ = 0, remaining_ = 0;
    };
}

namespace academy {
    Controller::Controller(Device& b) : board_(b) {
    }
    void Controller::tick() {
        const std::string cmd = board_.input.text;
        if (cmd != last_command_) {
            last_command_ = cmd;
            if (cmd == "start" && state_ == "idle") {
                state_ = "running";
                end_ = board_.input.now_ms + 1000;
                ++board_.output.events;
            } else if (cmd == "pause" && state_ == "running") {
                remaining_ = end_ - board_.input.now_ms;
                state_ = "paused";
                ++board_.output.events;
            } else if (cmd == "resume" && state_ == "paused") {
                end_ = board_.input.now_ms + remaining_;
                state_ = "running";
                ++board_.output.events;
            } else if (cmd == "reset") {
                state_ = "idle";
                remaining_ = 0;
                ++board_.output.events;
            }
        }
        if (state_ == "running" && board_.input.now_ms >= end_) {
            state_ = "done";
        }
        board_.output.state = state_;
        board_.output.led = state_ == "running";
        board_.output.reading = state_ == "running"
                                    ? end_ - board_.input.now_ms
                                    : (state_ == "paused" ? remaining_ : 0);
    }
}
